# 03 · 进阶：Checkpoint / HITL / Store / Time Travel

**对应章节**：LangGraph 教程第 03 章 —— 持久化（Checkpointer）、人工介入（HITL）、
长期记忆（Store）、以及基于状态历史的“时间旅行”（重试 / 分叉）。

**演示概念**：
- `MemorySaver` 作为 Checkpointer，让图可中断、可恢复、可回看历史；
- `interrupt` / `Command(resume=...)` 实现人在环路（HITL）；
- `InMemoryStore` 演示长期记忆的写入与检索（storedemo）；
- `get_state_history` / `update_state` 实现 Time Travel。

**运行前置**：
- 需要 `.env`（含有效 API Key），放在仓库根目录（与 .env.example 同位置）；
- 需要已 `uv sync`；
- 本 notebook 多个单元格会真实调用模型。

## 0) 长期记忆 Store 速览：`storedemo.py`

不依赖 LLM。演示 `InMemoryStore` 的 `put` / `search`，理解“长期记忆”和图状态（State）的区别：
State 是单次运行的临时状态，Store 是跨运行的持久记忆。

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
memory_ns = ("user-x","memory")

store.put(memory_ns,"mem1",{"joke_preference":"dark humor"})
store.put(memory_ns,"mem2",{"joke_dislike":"yellow"})

#print(store.get(memory_ns,"mem1"))
#print(store.search(memory_ns,query="joke_preference"))


print(store.search(memory_ns))

## 公共头部：引入 LLM 客户端

后续三个示例都依赖 `model`，从 `src.agent_cookbook` 引入。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # 仓库根目录，使 src 包可被导入
from src.agent_cookbook import model

## 1) 人工介入 HITL：`promptChainHITL.py`

依赖 `model`。用 `MemorySaver` 编译图；当重试次数达到上限，触发 `interrupt(...)` 把决策权
交给人。运行后会打印中断信息，再用 `Command(resume={"is_approved":True,...})` 恢复执行。

In [ ]:
from typing import TypedDict, Optional, Literal
from langgraph.graph import START, StateGraph, END
from langgraph.types import interrupt,Command
from langgraph.config import get_config
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()


class Joke(BaseModel):
    joke: str


class CriticResult(BaseModel):
    isFunny: bool = Field(description="is joke funny or not")
    opinion: str = Field(description="how to improve")



class AgentState(TypedDict):
    topic: str
    content: str
    reviewResult: Optional[CriticResult]
    humanReviewResult: Optional[CriticResult]
    retryCount: int


def genJoke(state: AgentState):
    message: str = f'gen a Joke about {state["topic"]}'
    if state["reviewResult"]:
        message = message + f', consider the opinion: {state["reviewResult"].opinion}'

    print(f'# gen \n gen message: {message}')
    response = model.chat.completions.create(
            response_model=Joke,
            messages=[{"role":"user","content":message}])
    return {"content": response.joke}


def reviewJoke(state: AgentState):
    # response = model.chat.completions.create(
    #     response_model=CriticResult,
    #     messages=[
    #         {"role":"system","content":"You are a strict joke reviewer."},
    #         {"role":"user","content":f'is this Joke funny or not ? Joke: {state["content"]},\
    #          if not let me know how to improve it'}
    #     ]
    # )
    #print(f'## review: \n is funny: {response.isFunny}, opinion: {response.opinion}')
    return {"reviewResult": {"isFunny":False,"opinion":"not funny at all!"}, "retryCount": state["retryCount"] + 1}


def humanReview(state: AgentState):
    config = get_config()
    max_retries = config.get("configurable", {}).get("max_retries", 3)

    interrupted = None
    if state["retryCount"] >= max_retries:
        print(f"⚠️ 达到最大重试次数 {max_retries}，need human approve")
        interrupted = interrupt({
            "question": f"How do think about this joke ,{state['content']}",
            "review": f"here is the review result, {state['reviewResult']}"
        })

    humanReviewResult = None
    if interrupted:
        humanReviewResult = {"humanReviewResult":CriticResult(isFunny=interrupted["is_approved"],opinion=interrupted["comment"])}
    
    return humanReviewResult

def decideNextStep(state: AgentState) -> Literal["genJoke","translate"]:
    if state["humanReviewResult"]:
        print(f'## human decision: {state["humanReviewResult"]}')
    if state["humanReviewResult"] and state["humanReviewResult"].isFunny or state["reviewResult"] and state["reviewResult"].isFunny:
        return "translate"
    
    return "genJoke"


def translate(state: AgentState):
    response = model.chat.completions.create(response_model=Joke,
                                             messages=[{"role":"user","content":f"translate to chinese: {state['content']}"}])
    return {"content": response.joke}


graph = StateGraph(AgentState).add_node("genJoke", genJoke)\
                              .add_node("review", reviewJoke)\
                              .add_node("translate", translate)\
                              .add_node("humanReview",humanReview)\
                              .add_edge(START, "genJoke")\
                              .add_edge("genJoke", "review")\
                              .add_edge("review","humanReview")\
                              .add_conditional_edges(
                                  "humanReview", decideNextStep,
                                  {"genJoke": "genJoke", "translate": "translate"})\
                              .add_edge("translate", END)

workflow = graph.compile(MemorySaver())
configs = RunnableConfig(configurable={"thread_id": "foo", "max_retries": 1})
result = workflow.invoke(
    {"topic":"wednesday","content":"","reviewResult":None,"humanReviewResult":None,"retryCount":0},
    config=configs,version="v2")

# from pprint import pprint 
# print(f'\n### channels:\n')
# pprint(workflow.channels)

#result["__interrupt__"]
if result.interrupts:
    print(f'\n###\ninterrupted: {result.interrupts}')
    resumed = workflow.invoke(Command(resume={"is_approved":True,"comment":"i think it's good"}), config={"configurable": {"thread_id": "foo", "max_retries": 1}})
    print(f'\n###\nresume result: {resumed}')
    final = resumed

## 2) HITL + Checkpoint 拦截：`promptChainWithStore.py`

依赖 `model`。把“是否需要人工确认”的判断放进 `checkReviewResult` 条件边里：
达到最大重试次数就 `interrupt`，根据人的 `is_approved` 决定去翻译还是继续重试。

In [ ]:
from typing import TypedDict, Optional, Literal
from langgraph.graph import START, StateGraph, END
from langgraph.types import interrupt,Command
from langgraph.config import get_config
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()


class Joke(BaseModel):
    joke: str


class CriticResult(BaseModel):
    isFunny: bool = Field(description="is joke funny or not")
    opinion: str = Field(description="how to improve")


class AgentState(TypedDict):
    topic: str
    content: str
    reviewResult: Optional[CriticResult]
    retryCount: int


def genJoke(state: AgentState):
    message: str = f'gen a Joke about {state["topic"]}'
    if state["reviewResult"]:
        message = message + f', consider the opinion: {state["reviewResult"].opinion}'

    print(f'# gen \n gen message: {message}')
    response = model.chat.completions.create(
            response_model=Joke,
            messages=[{"role":"user","content":message}])
    return {"content": response.joke}


def reviewJoke(state: AgentState):
    # response = model.chat.completions.create(
    #     response_model=CriticResult,
    #     messages=[
    #         {"role":"system","content":"You are a strict joke reviewer."},
    #         {"role":"user","content":f'is this Joke funny or not ? Joke: {state["content"]},\
    #          if not let me know how to improve it'}
    #     ]
    # )
    #print(f'## review: \n is funny: {response.isFunny}, opinion: {response.opinion}')
    return {"reviewResult": {"isFunny":False,"opinion":"not funny at all!"}, "retryCount": state["retryCount"] + 1}


def checkReviewResult(state: AgentState) -> Literal["genJoke", "translate"]:
    config = get_config()
    max_retries = config.get("configurable", {}).get("max_retries", 3)

    if state["retryCount"] >= max_retries:
        print(f"⚠️ 达到最大重试次数 {max_retries}，need human approve")
        interrupted = interrupt({
            "question": f"How do think about this joke ,{state['content']}",
            "review": f"here is the review result, {state['reviewResult']}"
        })

        print(f'\n human comment: {interrupted.get("comment","")}')
        if interrupted["is_approved"]:
            return "translate"
        else:
            return "genJoke"
    
    if state["reviewResult"] and state["reviewResult"].isFunny:
        print("✅ 评审通过")
        return "translate"

    print(f"❌ 评审未通过，重试第 {state['retryCount'] + 1}/{max_retries} 次")
    return "genJoke"


def translate(state: AgentState):
    response = model.chat.completions.create(response_model=Joke,
                                             messages=[{"role":"user","content":f"translate to chinese: {state['content']}"}])
    return {"content": response.joke}


graph = StateGraph(AgentState).add_node("genJoke", genJoke)\
                              .add_node("review", reviewJoke)\
                              .add_node("translate", translate)\
                              .add_edge(START, "genJoke")\
                              .add_edge("genJoke", "review")\
                              .add_conditional_edges(
                                  "review", checkReviewResult,
                                  {"genJoke": "genJoke", "translate": "translate"})\
                              .add_edge("translate", END)

workflow = graph.compile(MemorySaver())
result = workflow.invoke(
    {"topic":"wednesday","content":"","reviewResult":None,"retryCount":0},
    {"configurable": {"thread_id": "foo", "max_retries": 1}},version="v2")

#result["__interrupt__"]
if result.interrupts:
    print(f'\n###\ninterrupted: {result.interrupts}')
    resumed = workflow.invoke(Command(resume={"is_approved":True,"comment":"i think it's good"}), config={"configurable": {"thread_id": "foo", "max_retries": 1}})
    print(f'\n###\nresume result: {resumed}')
    final = resumed


#print(f'#### result: \n{result}')

## 3) 时间旅行 Time Travel：`promptChainHitlTimeTravel.py`

依赖 `model`。在 HITL 基础上，演示：
- `get_state_history` 回看每一次 checkpoint；
- 用 `update_state` 修改某个历史快照后重新 `invoke(None, snapshot.config)` 实现**重试（retry）**；
- 用 `update_state` + `invoke` 改写历史状态实现**分叉（fork）**。

运行会打印大量 checkpoint_id，对照代码理解“时间旅行”即可。

In [ ]:
from typing import TypedDict, Optional, Literal
from langgraph.graph import START, StateGraph, END
from langgraph.types import interrupt,Command
from langgraph.config import get_config
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()


class Joke(BaseModel):
    joke: str


class CriticResult(BaseModel):
    isFunny: bool = Field(description="is joke funny or not")
    opinion: str = Field(description="how to improve")


class AgentState(TypedDict):
    topic: str
    content: str
    reviewResult: Optional[CriticResult]
    humanReviewResult: Optional[CriticResult]
    retryCount: int


def genJoke(state: AgentState):
    message: str = f'gen a Joke about {state["topic"]}'
    if state["reviewResult"]:
        message = message + f', consider the opinion: {state["reviewResult"].opinion}'
    if state["humanReviewResult"]:
        message = message + f', consider human opinion: {state["humanReviewResult"].opinion}'

    print(f'# gen, round:{state["retryCount"] + 1} \n gen message: {message}')
    response = model.chat.completions.create(
            response_model=Joke,
            messages=[{"role":"user","content":message}])
    return {"content": response.joke}


def reviewJoke(state: AgentState):
    # response = model.chat.completions.create(
    #     response_model=CriticResult,
    #     messages=[
    #         {"role":"system","content":"You are a strict joke reviewer."},
    #         {"role":"user","content":f'is this Joke funny or not ? Joke: {state["content"]},\
    #          if not let me know how to improve it'}
    #     ]
    # )
    #print(f'## review: \n is funny: {response.isFunny}, opinion: {response.opinion}')
    return {"reviewResult": CriticResult(isFunny=False,opinion="not funny at all"), "retryCount": state["retryCount"] + 1}


def humanReview(state: AgentState):
    config = get_config()
    max_retries = config.get("configurable", {}).get("max_retries", 3)

    interrupted = None
    if state["retryCount"] >= max_retries:
        print(f"⚠️ 达到最大重试次数 {max_retries}，need human approve")
        interrupted = interrupt({
            "question": f"How do think about this joke ,{state['content']}",
            "review": f"here is the review result, {state['reviewResult']}"
        })

    humanReviewResult = None
    if interrupted:
        humanReviewResult = {"humanReviewResult":CriticResult(isFunny=interrupted["is_approved"],opinion=interrupted["comment"])}
    
    return humanReviewResult

def decideNextStep(state: AgentState) -> Literal["genJoke","translate"]:
    if state["humanReviewResult"]:
        print(f'## human decision: {state["humanReviewResult"]}')
    if state["humanReviewResult"] and state["humanReviewResult"].isFunny or state["reviewResult"] and state["reviewResult"].isFunny:
        return "translate"
    
    return "genJoke"
    


def translate(state: AgentState):
    response = model.chat.completions.create(response_model=Joke,
                                             messages=[{"role":"user","content":f"translate to chinese: {state['content']}"}])
    return {"content": response.joke}


graph = StateGraph(AgentState).add_node("genJoke", genJoke)\
                              .add_node("review", reviewJoke)\
                              .add_node("translate", translate)\
                              .add_node("humanReview",humanReview)\
                              .add_edge(START, "genJoke")\
                              .add_edge("genJoke", "review")\
                              .add_edge("review","humanReview")\
                              .add_conditional_edges(
                                  "humanReview", decideNextStep,
                                  {"genJoke": "genJoke", "translate": "translate"})\
                              .add_edge("translate", END)

workflow = graph.compile(MemorySaver())
configs = RunnableConfig(configurable={"thread_id": "foo", "max_retries": 1})

result = workflow.invoke(
    {"topic":"wednesday","content":"","reviewResult":None,"humanReviewResult":None,"retryCount":0},
    config=configs,version="v2")

#result["__interrupt__"]
import random

happy = 2 #random.randint(2,3)
print(f'\n### happy is {happy} ###\n')
attempt = 1
command = Command(resume={"is_approved":False,"comment":"i think it's not good"})
while result.interrupts:
    print(f'\n###\nattempt:{attempt}, interrupted: {result.interrupts}')
    if attempt == happy:
        command = Command(resume={"is_approved":True,"comment":"i think it's good"})

    result = workflow.invoke(command, 
                             config={"configurable": {"thread_id": "foo", "max_retries": 1}},
                             version="v2")
    attempt +=1
    

history = list(workflow.get_state_history(configs))
# History is in reverse chronological order
for state in history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}")

## 这个是retry
last_translate_index = next((i for i, s in enumerate(history) if 'translate' in s.next),None)
if last_translate_index:
    snapshot = history[last_translate_index-1]
    print(f"#### RETRY last translate:\ncheckpoint before first human review:\nnext={snapshot.next}, checkpoint_id={snapshot.config['configurable']['checkpoint_id'] }###\n")
    workflow.invoke(None,snapshot.config,version="v2")

retry_history = list(workflow.get_state_history(configs))
# History is in reverse chronological order
for state in retry_history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}")


## 这个是fork
reversed_history = list(reversed(history))  # 现在正序：最早 -> 最新
snapshotIndex = next((i for i, s in enumerate(reversed_history) if 'humanReview' in s.next),None)
if snapshotIndex:
    snapshot = reversed_history[snapshotIndex]
    fork_config = workflow.update_state(snapshot.config, {"humanReviewResult":CriticResult(isFunny=True,opinion="BIG funny"),"retryCount":1})
    print(f"\n\n\n#### Fork first HITL:\ncheckpoint before first human review:\nnext={snapshot.next}, checkpoint_id={snapshot.config['configurable']['checkpoint_id']}")
    workflow.invoke(None,fork_config,version="v2")

# Command(resume={"is_approved":True,"comment":"i think it's good"})
print(f'\n\n\n##### forked_history: ###\n')
retry_history = workflow.get_state_history(configs)
for state in retry_history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}")

#def print_history(workflow):

### 小结
- Checkpointer 是“可中断 / 可恢复 / 可回看”的基础；
- `interrupt` / `Command(resume=...)` 把人接入流程；
- Store 负责跨会话的长期记忆；
- `get_state_history` + `update_state` 让你能回到过去、改写分支，是调试与审计的利器。